# Test — pipeline complète de veille

Ce notebook lance tout le flux : lecture des requêtes Grist, récupération et nettoyage des amendements, filtrage, puis export dans Grist.

`gap=3` traite quatre jours : la date de référence et les trois jours précédents.

In [ ]:
import os

from B1.pipeline_complete import executer_pipeline_veilles


In [6]:
# Paramètres Grist
DOC_ID = "4pRcubnkqDJeXFtmwVKnkd"
TABLE_REQUETES = "Table1"  # identifiant technique de la table Grist des requêtes

# Évite de laisser la clé API dans le notebook ou dans Git.
# Sous PowerShell, avant de lancer Jupyter :
# $env:GRIST_API_KEY = "ta_cle"
API_KEY = os.environ["GRIST_API_KEY"]


In [5]:
import requests

BASE_URL = "https://grist.numerique.gouv.fr/o/docs/api"

r = requests.get(
    f"{BASE_URL}/docs/{DOC_ID}/tables",
    headers={"Authorization": f"Bearer {API_KEY}"},
)

r.raise_for_status()

for table in r.json()["tables"]:
    print(table["id"])

Table1


In [8]:
bilan = executer_pipeline_veilles(
    date_reference="2026-07-03",  # None pour utiliser la date locale du PC
    gap=0,                          # traite du 2026-07-01 au 2026-07-04 inclus
    doc_id=DOC_ID,
    api_key=API_KEY,
    table_requetes=TABLE_REQUETES,
    prefixe_sortie="Test_sortie",  # Test_sortie_veille_1, etc.
    mode_export="replace",         # remplace le contenu des sorties à chaque test
    ignorer_tables_vides=True,
    continuer_si_erreur=False,
    verbose=True,
)


Lecture des requêtes dans Grist : table 'Table1'...
2 veille(s) active(s) trouvée(s).
[2026-07-03] récupération et nettoyage des amendements...
[2026-07-03] 357 amendement(s) avant filtrage.


ValueError: Aucun filtre exploitable. Vérifie les mots-clés et les colonnes disponibles.

In [ ]:
print("Jours traités :", bilan["jours"])
print("Nombre de veilles actives :", bilan["nombre_veilles_actives"])
print("Lignes exportées par table :", bilan["tables"])

if bilan["erreurs_jours"]:
    print("Erreurs ignorées :", bilan["erreurs_jours"])


In [ ]:
# Inspecter les résultats locaux avant ou après l'export
for nom_veille, df in bilan["resultats"].items():
    print(f"\n{nom_veille} : {df.shape}")
    if not df.is_empty():
        display(df.head(5))


## Exécution quotidienne

Quand le test est validé, tu peux remplacer `prefixe_sortie="Test_sortie"` par `prefixe_sortie="Sortie"`. Avec `mode_export="replace"`, chaque exécution remplace les lignes précédentes dans les tables de sortie et évite les doublons.